Program 3: Obtain time-series stock-market data and perform analysis using Python by generating
candlestick charts, SMA and EMA overlays, rolling volatility bands, and volume trend
plots. Examine the data to identify trends, seasonal patterns, and significant changes
over time. Evaluate the visualizations to determine relationships among key variables,
variability, and long-term trends

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load the stock-market data
file_path = '/content/stocket_market/Market.csv'
try:
    df_raw = pd.read_csv(file_path)
    print("Data loaded successfully!")

    # Check if 'Index' column exists to filter distinct market indexes
    if 'Index' in df_raw.columns:
        indices = df_raw['Index'].unique()
        print("Available indices in dataset:", indices)
        # We default to 'NYA' (NYSE Composite Index) for a clean representation
        selected_index = 'NYA' if 'NYA' in indices else indices[0]
        print(f"Filtering data for index: {selected_index}")
        df = df_raw[df_raw['Index'] == selected_index].copy()
    else:
        df = df_raw.copy()

except Exception as e:
    print(f"Error loading CSV file: {e}")
    print("Generating fallback dummy stock data for demonstration...")
    dates = pd.date_range(start="2023-01-01", end="2023-12-31", freq="B")
    np.random.seed(42)
    close = 100 + np.cumsum(np.random.normal(0.1, 1.5, len(dates)))
    high = close + np.random.uniform(0.5, 3.0, len(dates))
    low = close - np.random.uniform(0.5, 3.0, len(dates))
    open_val = (high + low) / 2 + np.random.normal(0, 0.5, len(dates))
    volume = np.random.randint(100000, 1000000, len(dates))
    df = pd.DataFrame({
        'Date': dates,
        'Open': open_val,
        'High': high,
        'Low': low,
        'Close': close,
        'Volume': volume
    })

# 2. Preprocess Data
# Standardize column casing
rename_dict = {}
for col in df.columns:
    if col.lower() in ['date', 'open', 'high', 'low', 'close', 'volume']:
        rename_dict[col] = col.capitalize()
df.rename(columns=rename_dict, inplace=True)

# Convert Date and sort chronological
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date').reset_index(drop=True)

# Drop rows with NaN in important OHLC values to avoid calculation errors
df.dropna(subset=['Open', 'High', 'Low', 'Close'], inplace=True)
df.set_index('Date', inplace=True)

# Keep the last 250 trading days for a clear, readable chart layout
df = df.tail(250)

# 3. Calculate Technical Indicators
# Simple Moving Average (SMA) and Exponential Moving Average (EMA)
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

# Rolling Volatility Bands (Bollinger Bands: 20-day SMA +/- 2 standard deviations)
df['STD_20'] = df['Close'].rolling(window=20).std()
df['Upper_Band'] = df['SMA_20'] + (df['STD_20'] * 2)
df['Lower_Band'] = df['SMA_20'] - (df['STD_20'] * 2)

# 4. Generate Interactive Visualization using Plotly
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=('Candlestick Chart with SMA, EMA, and Volatility Bands', 'Volume Trend Plot'),
    row_width=[0.3, 0.7]
)

# Subplot 1: Candlestick Chart
fig.add_trace(
    go.Candlestick(
        x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
        name='Candlestick'
    ), row=1, col=1
)

# Overlays: SMA and EMA
fig.add_trace(
    go.Scatter(x=df.index, y=df['SMA_20'], line=dict(color='orange', width=1.5), name='SMA 20'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df.index, y=df['EMA_20'], line=dict(color='blue', width=1.5, dash='dash'), name='EMA 20'),
    row=1, col=1
)

# Overlays: Bollinger Volatility Bands
fig.add_trace(
    go.Scatter(x=df.index, y=df['Upper_Band'], line=dict(color='gray', width=1, dash='dot'), name='Upper Band (2 SD)'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df.index, y=df['Lower_Band'], line=dict(color='gray', width=1, dash='dot'), name='Lower Band (2 SD)'),
    row=1, col=1
)

# Subplot 2: Volume Trend Plot
fig.add_trace(
    go.Bar(x=df.index, y=df['Volume'], marker_color='purple', name='Volume'),
    row=2, col=1
)

# Update layout & styling
fig.update_layout(
    title_text='Advanced Stock Market Technical Analysis Dashboard',
    xaxis_rangeslider_visible=False,
    height=800,
    template='plotly_white',
    hovermode='x unified'
)
fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()

# 5. Output Summary Technical Insights
print("\n--- Technical Insights Summary ---")
last_close = df['Close'].iloc[-1]
last_sma = df['SMA_20'].iloc[-1]
last_upper = df['Upper_Band'].iloc[-1]
last_lower = df['Lower_Band'].iloc[-1]

print(f"Latest Close: {last_close:.2f}")
print(f"Latest 20-day SMA: {last_sma:.2f}")

if last_close > last_sma:
    print("- Trend: The asset is currently trading above its 20-day SMA, indicating a short-term bullish trend.")
else:
    print("- Trend: The asset is currently trading below its 20-day SMA, indicating a short-term bearish trend.")

if last_close > last_upper:
    print("- Volatility: Price has broken above the upper volatility band (potentially overbought).")
elif last_close < last_lower:
    print("- Volatility: Price has broken below the lower volatility band (potentially oversold).")
else:
    print("- Volatility: Pric`e remains inside the rolling volatility bands, showing normal market range.")


Data loaded successfully!
Available indices in dataset: ['NYA' 'IXIC' 'HSI' '000001.SS' 'GSPTSE' '399001.SZ' 'NSEI' 'GDAXI' 'KS11'
 'SSMI' 'TWII' 'J203.JO' 'N225' 'N100']
Filtering data for index: NYA



--- Technical Insights Summary ---
Latest Close: 16555.66
Latest 20-day SMA: 16381.96
- Trend: The asset is currently trading above its 20-day SMA, indicating a short-term bullish trend.
- Volatility: Price remains inside the rolling volatility bands, showing normal market range.
